# Zero-Call Trials & Export

无调用试次分析与结果导出。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from pathlib import Path

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

cluster_name_map = {
    0: '广泛试探型',
    1: '高效采纳型',
    2: '审慎批判型',
}

In [ ]:
# 输入文件路径
output_dir = Path('output')
trial_with_cluster_file = output_dir / 'trial_with_cluster.csv'

data_dir = Path('../../../data/analysis')
performance_file = data_dir / 'performance' / 'performance.csv'

In [ ]:
# 工具函数

# ---- GLMM 两两比较（基于固定效应估计，在对数链接尺度上）----
# 说明：针对前面拟合的 PoissonBayesMixedGLM（fluency 与 above_median），
# 计算不同 cluster 水平之间的两两对比：估计差值（link），标准误，z / p，
# 以及对数链接尺度的 95% 置信区间与 IRR（exp(diff)）区间。
import patsy
from itertools import combinations
from statsmodels.stats.multitest import multipletests
import numpy as np
from scipy import stats as _sps
import pandas as _pd

def pairwise_glmm_contrasts(glmm, glmm_res, df, formula='C(cluster) + dat_score + ribs_total + cse_total + ai_attitude', cluster_col='cluster', covariates=None, model_label='model'):
    exog_names = list(glmm.exog_names)
    k_fe = len(exog_names)

    # 获取固定效应及其协方差：VB 有 fe_mean / fe_sd，MAP 有 params / cov_params
    if hasattr(glmm_res, 'fe_mean'):
        fe_mean = np.asarray(glmm_res.fe_mean).reshape(-1)
        fe_sd = np.asarray(glmm_res.fe_sd).reshape(-1)
        cov_fe = np.diag(fe_sd ** 2)
        cov_fe = _pd.DataFrame(cov_fe, index=exog_names, columns=exog_names)
    else:
        all_params = np.asarray(glmm_res.params).reshape(-1)
        fe_mean = all_params[:k_fe]
        try:
            cov = np.asarray(glmm_res.cov_params())
            cov_fe = _pd.DataFrame(cov, index=glmm.exog_names, columns=glmm.exog_names)
            cov_fe = cov_fe.reindex(index=exog_names, columns=exog_names).fillna(0.0)
        except Exception:
            cov_fe = _pd.DataFrame(np.nan, index=exog_names, columns=exog_names)

    # cluster 水平
    if isinstance(df[cluster_col].dtype, pd.CategoricalDtype):
        clusters = df[cluster_col].cat.categories
    else:
        clusters = sorted(df[cluster_col].dropna().unique())

    pred_df = _pd.DataFrame({cluster_col: clusters})
    # 填充协变量（默认使用样本均值），covariates 可以传入 dict 覆盖
    if covariates is None:
        covariates = {}
    for cov, val in covariates.items():
        if val is None and cov in df.columns:
            pred_df[cov] = df[cov].mean()
        else:
            pred_df[cov] = val

    # 若连续协变量未传且存在于 df 中，则设为均值
    for cov in ['dat_score', 'ribs_total', 'cse_total', 'ai_attitude']:
        if cov not in pred_df.columns and cov in df.columns:
            pred_df[cov] = df[cov].mean()

    design = patsy.dmatrix(formula, pred_df, return_type='dataframe')
    # 保证 design 列与 exog_names 对齐
    for col in exog_names:
        if col not in design.columns:
            design[col] = 0.0
    design = design[exog_names]

    results = []
    for i, j in combinations(range(len(clusters)), 2):
        xa = design.iloc[i].values
        xb = design.iloc[j].values
        contrast = xa - xb
        diff = float(contrast.dot(fe_mean))
        # 计算方差（若协方差包含 NaN 则返回 NaN）
        try:
            var = float(contrast.dot(cov_fe.values).dot(contrast))
        except Exception:
            var = np.nan
        se = np.sqrt(var) if not np.isnan(var) else np.nan
        zstat = diff / se if (se and not np.isnan(se)) else np.nan
        pval = 2 * _sps.norm.sf(abs(zstat)) if not np.isnan(zstat) else np.nan
        # 95% CI 在 link（对数）尺度上，再转换到 IRR（exp）尺度
        if not np.isnan(se):
            ci_lo = diff - 1.96 * se
            ci_hi = diff + 1.96 * se
            irr = np.exp(diff)
            irr_lo = np.exp(ci_lo)
            irr_hi = np.exp(ci_hi)
        else:
            ci_lo = ci_hi = irr = irr_lo = irr_hi = np.nan
        results.append({
            'group1': clusters[i],
            'group2': clusters[j],
            'est_link': diff,
            'se': se,
            'z': zstat,
            'p': pval,
            'ci_lo_link': ci_lo,
            'ci_hi_link': ci_hi,
            'irr': irr,
            'irr_lo': irr_lo,
            'irr_hi': irr_hi
        })

    res_df = _pd.DataFrame(results).sort_values('p')
    if len(res_df):
        res_df['p_bonf'] = multipletests(res_df['p'].values, method='bonferroni')[1]
        res_df['p_fdr'] = multipletests(res_df['p'].values, method='fdr_bh')[1]
    print(f'--- {model_label} 两两比较（按 p 排序）---')
    display(res_df.round(4))
    return res_df


In [ ]:
# 加载数据
df_cluster = pd.read_csv(trial_with_cluster_file)  # 包含 participant_id, trial_index, cluster 与行为特征
df_perf = pd.read_csv(performance_file)             # 包含 participant_id, item_name, trial_index, fluency, originality

# 以试次为单位合并聚类标签
df_merged = pd.merge(
    df_perf,
    df_cluster,
    on=['participant_id', 'trial_index'],
    how='left'
 )

print(df_merged['cluster'].value_counts(dropna=False).sort_index())
df_merged.describe()

## 无调用试次

In [ ]:
df_merged[df_merged['cluster'] == -1][['fluency', 'originality']].describe()

# 对比有AI调用试次
df_merged[df_merged['cluster'] != -1][['fluency', 'originality']].describe()

In [ ]:
import statsmodels.api as sm

# 分组（直接用 trial_calls 而非 cluster）
call_trial_perf = df_merged[df_merged["trial_calls"] > 0][["fluency", "originality", "participant_id", "item_name"]].copy()
zero_trial_perf = df_merged[df_merged["trial_calls"] == 0][["fluency", "originality", "participant_id", "item_name"]].copy()
call_trial_perf["has_called"] = 1
zero_trial_perf["has_called"] = 0
df_compare = pd.concat([call_trial_perf, zero_trial_perf], ignore_index=True)
df_compare["participant_id"] = df_compare["participant_id"].astype("category")
df_compare["item_name"] = df_compare["item_name"].astype("category")

no_caller_ids = df_merged.groupby("participant_id")["trial_calls"].apply(lambda x: (x == 0).all())
no_caller_ids = no_caller_ids[no_caller_ids].index.tolist()

print(f"有调用试次: n={len(call_trial_perf)} (被试 {call_trial_perf['participant_id'].nunique()}, 物品 {call_trial_perf['item_name'].nunique()})")
print(f"无调用试次: n={len(zero_trial_perf)} (被试 {zero_trial_perf['participant_id'].nunique()}, 物品 {zero_trial_perf['item_name'].nunique()})")
print(f"完全未调用AI的被试: {len(no_caller_ids)} 名")
print()

print("=" * 60)
print("描述统计")
print("=" * 60)
for dv in ["fluency", "originality"]:
    c, z = call_trial_perf[dv], zero_trial_perf[dv]
    print(f"{dv}: 有调用 M={c.mean():.2f}(SD={c.std():.2f}), "
          f"无调用 M={z.mean():.2f}(SD={z.std():.2f}), dM={c.mean()-z.mean():.2f}")
print()

print("=" * 60)
print("LMM: has_called + (1|participant_id) + (1|item_name)")
print("=" * 60)
lmm_results = {}
for dv in ["fluency", "originality"]:
    model = sm.MixedLM.from_formula(
        f"{dv} ~ has_called", data=df_compare,
        groups="participant_id", vc_formula={"item_name": "1"}
    )
    result = model.fit(reml=False)  # ML以便与后文BIC模型比较保持估计方法一致
    fe, se, z, p = result.params, result.bse, result.tvalues, result.pvalues
    lmm_results[dv] = {
        "coef": round(float(fe["has_called"]), 4),
        "se": round(float(se["has_called"]), 4),
        "z": round(float(z["has_called"]), 4),
        "p": round(float(p["has_called"]), 6),
        "converged": result.converged
    }
    sig = "*" if p["has_called"] < .05 else "ns"
    print(f"{dv}: beta={lmm_results[dv]['coef']:.4f}, SE={lmm_results[dv]['se']:.4f}, "
          f"z={lmm_results[dv]['z']:.2f}, p={p['has_called']:.4f} {sig}")
    print(result.summary().tables[1])
print()

print("=" * 60)
print("Bayes Factor (BIC approximation)")
print("=" * 60)
bf_results = {}
for dv in ["fluency", "originality"]:
    m0 = sm.MixedLM.from_formula(f"{dv} ~ 1", data=df_compare,
        groups="participant_id", vc_formula={"item_name": "1"})
    r0 = m0.fit(reml=False)
    m1 = sm.MixedLM.from_formula(f"{dv} ~ has_called", data=df_compare,
        groups="participant_id", vc_formula={"item_name": "1"})
    r1 = m1.fit(reml=False)
    bic_diff = r0.bic - r1.bic
    bf10 = np.exp(bic_diff / 2)
    bf_results[dv] = {"bf10": round(bf10, 4), "delta_bic": round(bic_diff, 1)}
    if bf10 > 3:
        interp = "支持H1(有差异)"
    elif bf10 < 1/3:
        interp = "支持H0(无差异)"
    else:
        interp = "数据不明确"
    print(f"{dv}: deltaBIC={bic_diff:.1f}, BF10~{bf10:.2f} ({interp})")

print()
print("=" * 60)
print("汇总")
print("=" * 60)
for dv in ["fluency", "originality"]:
    print(f"\n{dv}:")
    print(f"  LMM: beta={lmm_results[dv]['coef']:.4f}, p={lmm_results[dv]['p']:.4f}")
    print(f"  BF10: ~{bf_results[dv]['bf10']:.2f}")

## 保存为json

In [ ]:
import json, itertools, numpy as np
from pathlib import Path
from scipy import stats as sp_stats
Path('output').mkdir(exist_ok=True)

# ── Correlations: behaviour x performance ──
behav_set = {'AI调用次数','首次调用时间','首次调用前想法数','调用前思考时间','观点采择率','受AI影响率'}
outcome_set = {'流畅性','原创性','高质量回答数'}
# 从 FDR 校正结果构造 pairs（p_labels 格式为 "label1  vs  label2"）
pairs = []
for label, r_val, p_raw in zip(p_labels, r_flat, p_flat):
    parts = label.split('  vs  ')
    if len(parts) == 2:
        pairs.append((parts[0], parts[1], r_val, p_raw))

corrs = []
for (bv, ov, r, p_raw), p_corr in zip(pairs, p_fdr):
    if bv in behav_set and ov in outcome_set:
        corrs.append({"bv": bv, "ov": ov, "r": round(float(r),4),
                      "p_raw": round(float(p_raw),6), "p_fdr": round(float(p_corr),6)})

# ── LMM originality ──
# Recreate prediction df since it was overwritten
_tmp_pred = pd.DataFrame({'cluster': lmm_df['cluster'].cat.categories})
_tmp_pred['participant_id'] = lmm_df['participant_id'].iloc[0]
_tmp_pred['item_name'] = lmm_df['item_name'].iloc[0]
_tmp_pred['dat_score'] = lmm_df['dat_score'].mean()
_tmp_pred['ribs_total'] = lmm_df['ribs_total'].mean()
_tmp_pred['cse_total'] = lmm_df['cse_total'].mean()
_tmp_pred['ai_attitude'] = lmm_df['ai_attitude'].mean()
_tmp_pred['pred_originality'] = lmm_res.predict(_tmp_pred)

lmm_preds = {str(int(row['cluster'])): round(float(row['pred_originality']),4)
             for _, row in _tmp_pred.iterrows()}
# res_df from cell 17 has pairwise comparison results
pw = []
for _, row in res_df.iterrows():
    pw.append({"c1": str(int(row['group1'])), "c2": str(int(row['group2'])),
               "diff": round(float(row['mean_diff']),6), "se": round(float(row['se']),6),
               "t": round(float(row['t']),4), "p": round(float(row['p']),6)})

wald = lmm_res.wald_test_terms(skip_single=False, scalar=True)
lmm_out = {
    "predicted_means": lmm_preds,
    "wald_chi2": round(float(wald.table.loc['C(cluster)','statistic']),4),
    "wald_p": round(float(wald.table.loc['C(cluster)','pvalue']),6),
    "pairwise": pw
}

# ── GLMM fluency ── (fluent_glmm_res is the fitted VB/MAP result)
if hasattr(fluent_glmm_res, 'fe_mean'):
    fm = np.asarray(fluent_glmm_res.fe_mean).reshape(-1)
    fs = np.asarray(fluent_glmm_res.fe_sd).reshape(-1)
else:
    fm = np.asarray(fluent_glmm_res.params).reshape(-1)[:len(fluent_glmm.exog_names)]
    fs = np.sqrt(np.diag(np.asarray(fluent_glmm_res.cov_params())))[:len(fluent_glmm.exog_names)]
flu_terms = []
for n, b, sd in zip(fluent_glmm.exog_names, fm, fs):
    if 'cluster' in n:
        flu_terms.append({"term": n, "B": round(float(b),4), "SD": round(float(sd),4),
                          "IRR": round(float(np.exp(b)),4),
                          "ci_lo": round(float(np.exp(b-1.96*sd)),4),
                          "ci_hi": round(float(np.exp(b+1.96*sd)),4)})

# ── GLMM above_median ── (ab_glmm_res is the fitted VB/MAP result)
if hasattr(ab_glmm_res, 'fe_mean'):
    afm = np.asarray(ab_glmm_res.fe_mean).reshape(-1)
    afs = np.asarray(ab_glmm_res.fe_sd).reshape(-1)
else:
    afm = np.asarray(ab_glmm_res.params).reshape(-1)[:len(ab_glmm.exog_names)]
    afs = np.sqrt(np.diag(np.asarray(ab_glmm_res.cov_params())))[:len(ab_glmm.exog_names)]
ab_terms = []
for n, b, sd in zip(ab_glmm.exog_names, afm, afs):
    if 'cluster' in n:
        ab_terms.append({"term": n, "B": round(float(b),4), "SD": round(float(sd),4),
                         "IRR": round(float(np.exp(b)),4),
                         "ci_lo": round(float(np.exp(b-1.96*sd)),4),
                         "ci_hi": round(float(np.exp(b+1.96*sd)),4)})

# ── Zero-call comparison ──
zc = {
    "call_n": int(len(call_trial_perf)),
    "call_flu_M": round(float(call_trial_perf['fluency'].mean()),4),
    "call_flu_SD": round(float(call_trial_perf['fluency'].std()),4),
    "call_orig_M": round(float(call_trial_perf['originality'].mean()),4),
    "call_orig_SD": round(float(call_trial_perf['originality'].std()),4),
    "zero_n": int(len(zero_trial_perf)),
    "zero_flu_M": round(float(zero_trial_perf['fluency'].mean()),4),
    "zero_flu_SD": round(float(zero_trial_perf['fluency'].std()),4),
    "zero_orig_M": round(float(zero_trial_perf['originality'].mean()),4),
    "zero_orig_SD": round(float(zero_trial_perf['originality'].std()),4),
    # LMM (crossed RE: participant_id + item_name)
    "lmm_fluency": lmm_results.get("fluency", {}),
    "lmm_originality": lmm_results.get("originality", {}),
    "bf_fluency": bf_results.get("fluency", {}),
    "bf_originality": bf_results.get("originality", {}),
}

r = {"correlations": corrs, "lmm_originality": lmm_out,
     "glmm_fluency": flu_terms, "glmm_above_median": ab_terms,
     "zero_call_comparison": zc}
Path('output/trial_performance_relation.json').write_text(json.dumps(r, ensure_ascii=False, indent=2))
print("Exported trial_performance_relation.json")